In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd
import numpy as np
import logging
import joblib
import pickle
import lmdb
from Bio import PDB
from Bio.PDB import PDBExceptions
from torch.utils.data import Dataset
from tqdm.auto import tqdm

import torch
import matplotlib.pyplot as plt
import seaborn as sns

import random

In [ ]:
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

def get_fasta_from_pdb(pdb_file):
    parser = PDBParser()
    structure = parser.get_structure("pdb", pdb_file)
    
    fasta_sequence = ""
    for chain in structure.get_chains():
        for residue in chain.get_residues():
                fasta_sequence += seq1(residue.get_resname())
    
    return fasta_sequence

In [ ]:
seq1('ASP')

In [ ]:
structure_dir = "/datapool/data2/home/jiahan/Data/PepMerge_new/"
seqs_dir = "/datapool/data2/home/jiahan/ResProj/PepDiff/frame-flow/Data/seqs"
bind_dic = torch.load("/datapool/data2/home/jiahan/ResProj/PepDiff/frame-flow/misc/affinity_dict.pt")

In [ ]:
all_pdbs = os.listdir(structure_dir)
print(len(all_pdbs))
all_pdbs = [x for x in all_pdbs if x in bind_dic]
print(len(all_pdbs))

In [ ]:
get_fasta_from_pdb(os.path.join(structure_dir,all_pdbs[0],'pocket.pdb'))

In [ ]:
with open(os.path.join(seqs_dir,'seqs.fasta'),'w') as f:
    for pdb in tqdm(all_pdbs):
        fasta = get_fasta_from_pdb(os.path.join(structure_dir,pdb,'receptor.pdb'))
        f.write(f'>{pdb}\n')
        f.write(fasta+'\n')
# mmseqs easy-cluster seqs.fasta clusterRes tmp --min-seq-id 0.4 -c 0.8 --cov-mode 1

In [ ]:
tab = pd.read_csv('/datapool/data2/home/jiahan/ResProj/PepDiff/frame-flow/Data/seqs/clusterRes_cluster.tsv',sep='\t',header=None)
tab.columns = ['center','id']
tab['cnts'] = tab.groupby('center')['id'].transform('count')
tab.sort_values('cnts',ascending=False,inplace=True)
tab

In [ ]:
cnts = tab.drop_duplicates('center')

In [ ]:
cnts[cnts['cnts']<5]['cnts'].sum()

In [ ]:
cnts[cnts['cnts']<5]['cnts'].sum()

In [ ]:
10384-2019

In [ ]:
tab.to_csv('/datapool/data2/home/jiahan/ResProj/PepDiff/frame-flow/Data/seqs/center.csv',index=None)

In [ ]:
len(set(tab['center']))

In [ ]:
len(set(tab['center']))

In [ ]:
cnts = pd.DataFrame(tab['center'].value_counts())
cnts = cnts.drop_duplicates(subset='center')
cnts.to_csv('/datapool/data2/home/jiahan/ResProj/PepDiff/frame-flow/Data/seqs/center.csv',index=None)

In [ ]:
samples = pd.read_csv("/datapool/data2/home/jiahan/Res Proj/PepDiff/frame-flow/misc/231220/sample_all.csv")
samples

In [ ]:
res = pd.merge(tab,samples,on='id')
# res[['center','id','cnts','len']].to_csv('/datapool/data2/home/jiahan/Res Proj/PepDiff/frame-flow/Data/seqs/meta_data.csv',index=False)
res.to_csv('/datapool/data2/home/jiahan/Res Proj/PepDiff/frame-flow/Data/seqs/meta_data.csv',index=False)

In [ ]:
res = pd.read_csv('/datapool/data2/home/jiahan/Res Proj/PepDiff/frame-flow/Data/seqs/meta_data.csv')
res

In [ ]:
centers = set((res[(res['cnts']>=10)&(res['cnts']<=100)])['center'])
len(centers)

In [ ]:
tests = random.sample(centers, 10)

In [ ]:
tmp = res[res['center'].isin(tests)]
tmp

In [ ]:
tmp['tran'].mean()

In [ ]:
with open("/datapool/data2/home/jiahan/Res Proj/PepDiff/frame-flow/Data/RF_samples/names.txt",'w') as f:
    for i,row in tmp.iterrows():
        f.write(row['id']+'\n')